In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt


# ====== 1) 경로 설정 ======
train_dir = r"C:\Users\itwill\Desktop\python\딥러닝\deep_test_2\동물\검증용"
val_dir = r"C:\Users\itwill\Desktop\python\딥러닝\deep_test_2\동물\학습용"


# ====== 2) 하이퍼파라미터 ======
IMG_SIZE = (224, 224)      # 너무 크면 느려짐 / 너무 작으면 성능 저하됨
BATCH_SIZE = 8
SEED = 42


# ====== 3) 데이터 로드 (폴더 이름이 라벨이 됨) ======
# 학습용 이미지
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\itwill\자동화 공부\tf311\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\itwill\자동화 공부\tf311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: DLL 초기화 루틴을 실행할 수 없습니다.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [ ]:
# 검증용 이미지
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("클래스:", class_names)

In [ ]:
import sys
print(sys.version)

import tensorflow as tf
print(tf.__version__)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

[TensorFlow DLL Diagnostic] Analyzing: c:\Users\itwill\자동화 공부\tf311\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\itwill\자동화 공부\tf311\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: DLL 초기화 루틴을 실행할 수 없습니다.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

In [7]:
!pip install opencv-python

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   ----------- ---------------------------- 11.3/40.2 MB 88.1 MB/s eta 0:00:01
   ------------------------------- -------- 31.5/40.2 MB 86.7 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 73.0 MB/s  0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 12.6/12.6 MB 71.8 MB/s  0:00:00

  Attempting uninstall: numpy

    Found existing installation: numpy 1.26.4

   ---------------------------------------- 0/2 [numpy]
    Uninstalling numpy-1.26.4:
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.4.6 which is incompatible.


In [8]:
# 동영상에서 찾고 싶은 얼굴을 샘플 이미지로 추출하기
import cv2
import os

# ===== 설정 =====
VIDEO_PATH = r"C:\Users\itwill\Desktop\python\딥러닝\류현진.mp4"
SAVE_DIR = r"C:\Users\itwill\Desktop\python\딥러닝\결과"
MAX_COUNT = 200
SHOW_WIN = True

os.makedirs(SAVE_DIR, exist_ok=True)

# ===== 비디오 로드 =====
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"비디오를 열 수 없습니다: {VIDEO_PATH}")

# ===== 얼굴 검출기 =====
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_classifier.empty():
    raise RuntimeError("Haar cascade 로드 실패.")

def face_extractor(img):
    """
    프레임에서 가장 큰 얼굴 1개를 잘라서 반환.
    얼굴이 없으면 None 반환.
    """
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_classifier.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(60, 60)
    )

    if len(faces) == 0:
        return None

    # 여러 얼굴 중 가장 큰 얼굴 선택
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])

    # 얼굴 영역 crop
    face = img[y:y + h, x:x + w]

    return face

count = 0
frame_count = 0
font = cv2.FONT_HERSHEY_COMPLEX

while True:
    ret, frame = cap.read()

    if not ret:
        print("영상 끝 또는 프레임 읽기 실패")
        break

    frame_count += 1

    face = face_extractor(frame)

    if face is not None:
        count += 1

        face = cv2.resize(face, (200, 200))

        save_path = os.path.join(SAVE_DIR, f"face_{count:03d}.jpg")
        cv2.imwrite(save_path, face)

        cv2.putText(
            face,
            str(count),
            (10, 30),
            font,
            1,
            (0, 255, 0),
            2
        )

        if SHOW_WIN:
            cv2.imshow("Extracted Face", face)

    if SHOW_WIN:
        cv2.imshow("Original Video", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            print("사용자 종료")
            break

    if count >= MAX_COUNT:
        print(f"얼굴 이미지 {MAX_COUNT}장 저장 완료")
        break

cap.release()

if SHOW_WIN:
    cv2.destroyAllWindows()

print(f"총 읽은 프레임 수: {frame_count}")
print(f"저장된 얼굴 이미지 수: {count}")
print(f"저장 폴더: {SAVE_DIR}")

얼굴 이미지 200장 저장 완료
총 읽은 프레임 수: 226
저장된 얼굴 이미지 수: 200
저장 폴더: C:\Users\itwill\Desktop\python\딥러닝\결과


In [13]:
# 동영상에서 찾고 싶은 얼굴을 샘플 이미지로 추출하기
import cv2
import os

# ===== 설정 =====
VIDEO_PATH = r"C:\Users\itwill\Desktop\python\딥러닝\류현진.mp4"
SAVE_DIR = r"C:\Users\itwill\Desktop\python\image"  # 샘플 이미지를 저장할 폴더
MAX_COUNT = 200  # 샘플로 학습용으로 저장할 얼굴 이미지 수
SHOW_WIN = True  # imshow가 안 되는 환경이면 False로 바꾸세요

os.makedirs(SAVE_DIR, exist_ok=True)

# ===== 비디오 로드 =====
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"비디오를 열 수 없습니다: {VIDEO_PATH}")

# ===== 얼굴 검출기(가장 안전한 내장 경로 사용) =====
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

if face_classifier.empty():
    raise RuntimeError("Haar cascade 로드 실패.")

def face_extractor(img):
    """프레임에서 가장 큰 얼굴 1개를 잘라서 반환. 없으면 None."""
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_classifier.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(60, 60)
    )

    48

    # 얼굴 없으면 빈 배열 -> len(faces)==0
    if len(faces) == 0:
        return None

    # 여러 얼굴 중 가장 큰 얼굴 선택
    x, y, w, h = max(faces, key=lambda b: b[2] * b[3])

    # crop
    return img[y:y+h, x:x+w]

count = 0
font = cv2.FONT_HERSHEY_COMPLEX

while True:
    ret, frame = cap.read()

    if not ret:
        print("영상 끝 또는 프레임 읽기 실패")
        break

    # face_extractor를 한 번만 호출
    face_img = face_extractor(frame)

    if face_img is not None:
        count += 1

        # 200x200 리사이즈 + 그레이 변환
        face = cv2.resize(face_img, (200, 200))
        face_gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)

        # 저장
        file_name_path = os.path.join(SAVE_DIR, f"user{count}.jpg")
        cv2.imwrite(file_name_path, face_gray)

        # 화면 출력(가능한 환경일 때만)
        if SHOW_WIN:
            view = face_gray.copy()
            cv2.putText(view, str(count), (50, 50), font, 1, (255, 255, 255), 2)
            cv2.imshow("Face Cropper", view)

    49

    # 종료 조건: Enter(13) 또는 MAX_COUNT 도달
    if SHOW_WIN:
        key = cv2.waitKey(1)

        if key == 13 or count >= MAX_COUNT:
            break
    else:
        if count >= MAX_COUNT:
            break

cap.release()

if SHOW_WIN:
    cv2.destroyAllWindows()

print(f"학습할 이미지 저장 완료: {count}장 -> {SAVE_DIR}")

학습할 이미지 저장 완료: 200장 -> C:\Users\itwill\Desktop\python\image


In [11]:
cv2.imwrite(file_name_path, face_gray)

False

In [12]:
saved = cv2.imwrite(file_name_path, face_gray)
print("저장 경로:", file_name_path)
print("저장 성공:", saved)
print("파일 존재:", os.path.exists(file_name_path))

저장 경로: C:\Users\itwill\Desktop\python\딥러닝\결과\user200.jpg
저장 성공: False
파일 존재: False


In [20]:
# 샘플 이미지를 학습한 후 동영상에서 해당 이미지만 찾아내기
import os
from os.path import isfile, join

import numpy as np
import cv2

print('샘플 학습 이미지를 찾아옵니다!!!')

data_path = r"C:\Users\itwill\Desktop\python\image\RYU"

# 이미지 파일만 가져오기 (mp4/다른 파일 섞이면 imread가 None이 됩니다)
onlyfiles = [
    f for f in os.listdir(data_path)
    if isfile(join(data_path, f)) and f.lower().endswith(('.jpg', '.jpeg', '.png'))
]

Training_Data, Labels = [], []
label = 0

for f in onlyfiles:
    image_path = join(data_path, f)  # 경로 안전 결합
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    # 읽기 실패(None)면 스킵 + 원인 확인용 출력
    if img is None:
        print("읽기 실패(스킵):", image_path)
        continue

    Training_Data.append(img.astype(np.uint8))
    Labels.append(label)
    label += 1

if len(Training_Data) == 0:
    raise RuntimeError("학습할 이미지가 0장입니다. (저장된 얼굴 jpg가 있는지 확인)")

Labels = np.asarray(Labels, dtype=np.int32)

51

# LBPH는 opencv-contrib-python 필요
# pip install opencv-contrib-python
model = cv2.face.LBPHFaceRecognizer_create()
model.train(np.asarray(Training_Data), Labels)

print("Model 학습 완료함!!!!!")
print("학습 이미지 수:", len(Training_Data))

샘플 학습 이미지를 찾아옵니다!!!


AttributeError: module 'cv2' has no attribute 'face'

In [18]:
import cv2

print(cv2.__version__)
print(hasattr(cv2, "face"))
print(dir(cv2.face) if hasattr(cv2, "face") else "cv2.face 없음")

4.13.0
False
cv2.face 없음


In [19]:
import sys

print("현재 Jupyter 파이썬 경로:")
print(sys.executable)

!{sys.executable} -m pip uninstall opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless -y
!{sys.executable} -m pip install opencv-contrib-python

현재 Jupyter 파이썬 경로:
c:\Users\itwill\AppData\Local\Programs\Python\Python313\python.exe
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92


You can safely remove it manually.


  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl (46.5 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import sys

print(sys.executable)

!{sys.executable} -m pip uninstall opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless -y
!{sys.executable} -m pip install opencv-contrib-python

c:\Users\itwill\AppData\Local\Programs\Python\Python313\python.exe
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92


  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl (46.5 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
import sys

print("현재 파이썬:", sys.executable)

!{sys.executable} -m pip uninstall opencv-python opencv-contrib-python opencv-python-headless opencv-contrib-python-headless -y
!{sys.executable} -m pip install opencv-contrib-python

현재 파이썬: c:\Users\itwill\AppData\Local\Programs\Python\Python313\python.exe
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92


  Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_contrib_python-4.13.0.92-cp37-abi3-win_amd64.whl (46.5 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
!pip install paddleocr
!pip install paddlepaddle

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.3.5 which is incompatible.


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 41.9 MB/s  0:00:00
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   -------------------------------- ------- 10.5/13.1 MB 59.4 MB/s eta 0:00:01
   ---------------------------------------- 13.1/13.1 MB 39.1 MB/s  0:00:00
   ---------------------------------------- 0.0/45.5 MB ? eta -:--:--
   ------------ --------------------------- 14.7/45.5 MB 65.8 MB/s eta 0:00:01
   -------------------- ------------------- 23.1/45.5 MB 56.0 MB/s eta 0:00:01
   ----------------------------- ---------- 34.1/45.5 MB 52.8 MB/s eta 0:00:01
   ---------------------------------------  45.4/45.5 MB 57.7 MB/s eta 0:00:01
   ---------------------------------------- 45.5/45.5 MB 45.3 MB/s  0:00:01
   ---------------------------------------- 0.0/6.1 MB ? eta -:--:--
   ---------------------------------------- 6.1/6.1 MB 46.6 MB/s  0:00:00
   ------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.1 requires numpy<2.0.0,>=1.23.5; python_version <= "3.11", but you have numpy 2.3.5 which is incompatible.


In [25]:
import os

img_path = r"C:\Users\itwill\Desktop\python\딥러닝\영수증.png"

print("이미지 경로:", img_path)
print("파일 존재 여부:", os.path.exists(img_path))

이미지 경로: C:\Users\itwill\Desktop\python\딥러닝\영수증.png
파일 존재 여부: True


In [30]:
import numpy as np
from paddleocr import PaddleOCR
def poly_to_features(poly):
 pts = np.array(poly, dtype=np.float32)
 xs, ys = pts[:, 0], pts[:, 1]
 y_center = float(np.mean(ys))
 x_min = float(np.min(xs))
 return y_center, x_min

In [29]:
import sys

print("현재 Jupyter 파이썬:", sys.executable)

!{sys.executable} -m pip install paddleocr
!{sys.executable} -m pip install paddlepaddle

현재 Jupyter 파이썬: c:\Users\itwill\AppData\Local\Programs\Python\Python313\python.exe
  Using cached paddleocr-3.6.0-py3-none-any.whl.metadata (26 kB)
  Using cached paddlex-3.6.1-py3-none-any.whl.metadata (80 kB)
  Using cached aistudio_sdk-0.3.8-py3-none-any.whl.metadata (1.1 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached modelscope-1.37.1-py3-none-any.whl.metadata (43 kB)
  Using cached prettytable-3.17.0-py3-none-any.whl.metadata (34 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
  Using cached ruamel_yaml-0.19.1-py3-none-any.whl.metadata (16 kB)
  Using cached imagesize-2.0.0-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached pypdfium2-5.9.0-py3-none-win_amd64.whl.metadata (68 kB)
  Using cached bce_python_sdk-0.9.71-py3-none-any.whl.metadata (449 bytes)
  Using cached pycryptodome-3.23.0-cp37-abi3-win_amd64.whl.metadata (3.5 kB)
 

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached opt_einsum-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/104.8 MB ? eta -:--:--
   -- ------------------------------------- 6.3/104.8 MB 65.4 MB/s eta 0:00:02
   ---------- ----------------------------- 28.0/104.8 MB 88.8 MB/s eta 0:00:01
   ----------------- ---------------------- 46.1/104.8 MB 88.7 MB/s eta 0:00:01
   ------------------------ --------------- 64.0/104.8 MB 90.5 MB/s eta 0:00:01
   --------------------------- ------------ 72.6/104.8 MB 78.5 MB/s eta 0:00:01
   --------------------------------- ------ 87.0/104.8 MB 81.7 MB/s eta 0:00:01
   ---------------------------------- ----- 91.5/104.8 MB 67.5 MB/s eta 0:00:01
   --------------------------------------  102.8/104.8 MB 67.7 MB/s eta 0:00:01
   --------------------------------------  103.3/104.8 MB 59.1 MB/s eta 0:00:01
   --------------------------------------  104.6/104.8 MB 56.2 MB/s 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
def cluster_into_lines(items, y_tol=14):
    """
    items: [{'text':..., 'y': y_center, 'x': x_min}, ...]
    y_tol: 같은 줄로 볼 y중심 허용 오차(픽셀). 필요하면 10~25로 조정
    """
    items = sorted(items, key=lambda it: it["y"])
    lines = []
    cur = []
    cur_y = None

    for it in items:
        if not cur:
            cur = [it]
            cur_y = it["y"]
            continue

        if abs(it["y"] - cur_y) <= y_tol:
            cur.append(it)

            # 줄 중심 업데이트(견고)
            cur_y = (cur_y * (len(cur) - 1) + it["y"]) / len(cur)
        else:
            cur.sort(key=lambda x: x["x"])
            lines.append(cur)

            cur = [it]
            cur_y = it["y"]

    if cur:
        cur.sort(key=lambda x: x["x"])
        lines.append(cur)

    return lines

In [32]:
img_path = r"C:\Users\itwill\Desktop\python\딥러닝\영수증.png"

ocr = PaddleOCR(
    lang="korean",
    use_textline_orientation=True
)

result = ocr.predict(img_path)

print("OCR 실행 완료")
print("결과 타입:", type(result))
print("결과 개수:", len(result))

c:\Users\itwill\AppData\Local\Programs\Python\Python313\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `C:\Users\itwill\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `C:\Users\itwill\.paddlex\official_models\UVDoc`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `C:\Users\itwill\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.


...

Creating model: ('PP-OCRv5_server_det', None, None)
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `C:\Users\itwill\.paddlex\official_models\PP-OCRv5_server_det`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('korean_PP-OCRv5_mobile_rec', None, None)
Using official model (korean_PP-OCRv5_mobile_rec), the model files will be automatically downloaded and saved in `C:\Users\itwill\.paddlex\official_models\korean_PP-OCRv5_mobile_rec`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)


In [33]:
import os

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["FLAGS_enable_pir_api"] = "0"

print("Paddle oneDNN/MKLDNN 비활성화 설정 완료")

Paddle oneDNN/MKLDNN 비활성화 설정 완료


In [34]:
import numpy as np
from paddleocr import PaddleOCR

img_path = r"C:\Users\itwill\Desktop\python\딥러닝\영수증.png"

ocr = PaddleOCR(
    lang="korean",
    use_textline_orientation=True
)

result = ocr.predict(
    img_path,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True
)

print("OCR 실행 완료")
print("결과 타입:", type(result))
print("결과 개수:", len(result))

if len(result) > 0:
    print("첫 번째 결과 타입:", type(result[0]))

Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\itwill\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('korean_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directo

NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at ..\paddle\fluid\framework\new_executor\instruction\onednn\onednn_instruction.cc:118)


In [1]:
python -m ipykernel install --user --name receipt_ocr_env --display-name "Python 3.11 receipt_ocr_env"

SyntaxError: invalid syntax (750409295.py, line 1)

In [ ]:
import sys

print(sys.executable)